# Cadence PII Detection — live walkthrough

Upload the three **synthetic teaching artifacts** from the case study. This notebook runs extraction, OCR and PII detection inside the active Python runtime; it makes no document-content API calls. Google Colab is cloud hosted, so uploaded files do leave your computer. If your policy disallows that, run this notebook locally in Jupyter.

Run top to bottom. If setup upgrades Pillow in an already-running Colab session, restart the session and choose Run all again; the installed packages remain in place. Review-required files are withheld from the downstream AI boundary until a person checks the source images and signs off. Redaction is pseudonymisation, and a zero-residual scan does not prove that no PII remains.

In [ ]:
import importlib.metadata as metadata
import os, shutil, subprocess, sys
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
pillow_loaded_before_install = getattr(sys.modules.get('PIL'), '__version__', None)
if IN_COLAB:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'libgl1'], check=True)
    packages = ['docling==2.138.0', 'presidio-analyzer==2.2.364', 'presidio-anonymizer==2.2.364',
                'spacy==3.8.16', 'pymupdf==1.28.2', 'python-docx==1.2.0', 'python-pptx==1.0.2',
                'pytesseract==0.3.13', 'Pillow==12.3.0', 'pandas==3.0.6']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_lg'], check=True)
    # Colab may have imported an older Pillow before pip upgraded it. Mixed in-memory
    # and on-disk PIL modules can fail with `cannot import name _Ink`.
    stale_pillow_typing = sys.modules.get('PIL._typing')
    if ((pillow_loaded_before_install and pillow_loaded_before_install != metadata.version('Pillow'))
            or (stale_pillow_typing is not None and not hasattr(stale_pillow_typing, '_Ink'))):
        raise RuntimeError('Pillow was upgraded in this session. Choose Runtime > Restart session, then Run all again. Packages are already installed.')
for package in ['docling', 'presidio-analyzer', 'spacy', 'python-pptx', 'pytesseract']:
    print(package, metadata.version(package))
subprocess.run(['tesseract', '--version'], check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
Path('cadence_pii/detect').mkdir(parents=True, exist_ok=True)
Path('cadence_pii/extract').mkdir(parents=True, exist_ok=True)

## Pipeline implementation

These cells write the exact modules used by the local tests. No separate repository download is needed in Colab.

In [ ]:
%%writefile cadence_pii/__init__.py
"""Cadence PII detection pipeline (Optiv Case Study 2).

Flow: ingest -> extract (+OCR) -> detect -> merge -> redact -> verify -> gate -> report.
Everything runs inside the local Python runtime; no document content is sent to external services.
"""
__version__ = "0.1.0"

In [ ]:
%%writefile cadence_pii/config.py
"""Central configuration: entity taxonomy, thresholds and feature toggles."""
from dataclasses import dataclass, field

# Entity taxonomy. Placeholder prefix is what appears in redacted text, e.g. [PERSON_3].
ENTITY_TYPES = {
    "PERSON": "Full or partial personal name",
    "EMAIL": "E-mail address",
    "PHONE": "Telephone / mobile number",
    "EMPLOYEE_ID": "Employee / vendor-personnel identifier (e.g. EMP-38104, DIR-0063, MER-IN-4471)",
    "US_SSN": "US Social Security number (full or last-four reference)",
    "PASSPORT": "Passport number",
    "TAX_ID": "Tax identifier (e.g. Indian PAN)",
    "NATIONAL_ID": "National identity number (e.g. Polish PESEL, UK NINO, Aadhaar)",
    "DATE_OF_BIRTH": "Date of birth",
    "ADDRESS": "Street / home address",
    "FINANCIAL_ACCOUNT": "Card or account number / fragment (e.g. 'card ending 2218')",
}

# Roll-up used in audit summaries (the brief asks specifically about government-issued IDs).
CATEGORY = {"US_SSN": "GOVERNMENT_ID", "PASSPORT": "GOVERNMENT_ID", "TAX_ID": "GOVERNMENT_ID",
            "NATIONAL_ID": "GOVERNMENT_ID", "EMAIL": "CONTACT", "PHONE": "CONTACT", "ADDRESS": "CONTACT",
            "PERSON": "NAME", "EMPLOYEE_ID": "EMPLOYEE_ID", "DATE_OF_BIRTH": "DATE_OF_BIRTH",
            "FINANCIAL_ACCOUNT": "FINANCIAL"}

# Presidio entity names mapped onto our taxonomy (anything not listed is ignored).
PRESIDIO_MAP = {
    "PERSON": "PERSON",
    "EMAIL_ADDRESS": "EMAIL",
    "PHONE_NUMBER": "PHONE",
    "US_SSN": "US_SSN",
    "US_PASSPORT": "PASSPORT",
    "IN_PAN": "TAX_ID",
    "IN_AADHAAR": "NATIONAL_ID",
    "UK_NINO": "NATIONAL_ID",
    "CREDIT_CARD": "FINANCIAL_ACCOUNT",
    "IBAN_CODE": "FINANCIAL_ACCOUNT",
    "US_BANK_NUMBER": "FINANCIAL_ACCOUNT",
}

# GLiNER labels mapped onto our taxonomy.
GLINER_MAP = {
    "person": "PERSON",
    "email": "EMAIL",
    "phone number": "PHONE",
    "social security number": "US_SSN",
    "passport number": "PASSPORT",
    "tax identification number": "TAX_ID",
    "date of birth": "DATE_OF_BIRTH",
    "address": "ADDRESS",
}

# Terms that NER models commonly mis-tag as PERSON in this corpus (organisations, products, roles).
# Matching is case-insensitive on the whole detected span.
PERSON_DENYLIST = {
    "cadence", "cadence financial group", "onetrust", "optiv", "board risk committee", "tprm",
    "rcsa", "kri", "kci", "raci", "ceo", "cfo", "coo", "cro", "cio", "dpo", "soc", "sla",
}


# Generic business / UI vocabulary that NER mislabels as PERSON in policy and configuration documents.
COMMON_WORDS = {
    "workflow", "policy", "vendor", "tier", "email", "cloud", "reviewer", "data", "approval", "owner", "risk",
    "template", "questionnaire", "attribute", "stage", "rule", "rules", "allowed", "collected", "please",
    "specify", "the", "notify", "reject", "launching", "send", "push", "peer", "setup", "admin", "user",
    "users", "group", "groups", "manager", "inventory", "engagement", "assessment", "contract", "service",
    "provider", "partner", "onboarding", "offboarding", "review", "status", "submit", "approve", "task",
    # document structure / rating vocabulary
    "appendix", "section", "figure", "table", "page", "chapter", "annex", "schedule", "part", "contents",
    "critical", "high", "medium", "low", "moderate", "elevated", "severe", "minor", "major", "unlikely",
    "likely", "rare", "almost", "certain", "possible", "biennial", "annual", "quarterly", "monthly",
    "continuous", "anonymous", "legal", "compliance", "uncontrolled", "printed", "internal", "confidential",
    "chief", "officer", "director", "head", "pillar", "actionability",
}


@dataclass
class Settings:
    use_presidio: bool = True
    use_gliner: bool = False              # optional layer; enable only if evaluation shows benefit
    gliner_model: str = "urchade/gliner_multi_pii-v1"
    gliner_threshold: float = 0.5
    presidio_threshold: float = 0.5       # minimum Presidio score kept in the primary pass
    verify_presidio_threshold: float = 0.35  # stricter (lower) threshold in the verification pass
    ocr_lang: str = "eng"
    ocr_min_block_conf: float = 60.0      # OCR blocks below this mean confidence -> human review
    ocr_min_words_for_conf: int = 5       # ignore confidence on tiny fragments (logos, icons)
    pdf_dual_ocr: bool = True             # independent full-resolution Tesseract view for PDF cross-check
    redact_all_dates: bool = False        # policy choice: only birth dates are personal by default
    max_file_mb: int = 100
    allowed_ext: tuple = (".pdf", ".docx", ".pptx")
    extra: dict = field(default_factory=dict)


DEFAULT = Settings()

In [ ]:
%%writefile cadence_pii/detect/__init__.py
"""Detection layers. Each returns a list[Finding] with offsets into the text it was given."""

In [ ]:
%%writefile cadence_pii/detect/gliner_det.py
"""Optional GLiNER layer (zero-shot NER, runs locally once weights are downloaded).

Disabled by default; enabled only if the evaluation shows a recall gain worth the runtime.
"""
from __future__ import annotations

from functools import lru_cache

from ..config import GLINER_MAP
from ..models import Finding


@lru_cache(maxsize=1)
def get_model(name: str):
    from gliner import GLiNER
    return GLiNER.from_pretrained(name)


def detect_gliner(text: str, offset: int, model_name: str, threshold: float = 0.5, window: int = 1200) -> list:
    """GLiNER has a limited context window, so long blocks are scanned in overlapping chunks."""
    if not text.strip():
        return []
    model = get_model(model_name)
    labels = list(GLINER_MAP)
    out, seen = [], set()
    step = window - 200
    for cs in range(0, max(1, len(text)), step):
        chunk = text[cs:cs + window]
        for ent in model.predict_entities(chunk, labels, threshold=threshold):
            s, e = ent["start"] + cs, ent["end"] + cs
            if (s, e) in seen:
                continue
            seen.add((s, e))
            out.append(Finding(GLINER_MAP[ent["label"]], s + offset, e + offset, text[s:e],
                               round(float(ent["score"]), 3), "gliner", "gliner:" + ent["label"]))
        if cs + window >= len(text):
            break
    return out

In [ ]:
%%writefile cadence_pii/detect/merge.py
"""Filtering, deterministic overlap resolution and within-document name propagation."""
from __future__ import annotations

import re
from functools import lru_cache

from ..config import COMMON_WORDS, PERSON_DENYLIST
from ..models import Finding

METHOD_PRIORITY = {"regex": 3, "presidio": 2, "gliner": 1, "propagation": 1, "cross_view": 1}
# When spans of different types overlap, structured identifiers win over generic NER types.
TYPE_PRIORITY = {"EMAIL": 9, "US_SSN": 8, "PASSPORT": 8, "TAX_ID": 8, "FINANCIAL_ACCOUNT": 7, "EMPLOYEE_ID": 7,
                 "PHONE": 6, "DATE_OF_BIRTH": 6, "ADDRESS": 5, "PERSON": 4}


_EMAIL_RX = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")


def lowercase_vocab(text: str) -> set:
    """Words that occur in lowercase somewhere in the document prose (e-mail addresses excluded)."""
    return set(re.findall(r"\b[a-z][a-z'’-]+\b", _EMAIL_RX.sub(" ", text)))


def _name_tokens(v: str) -> list:
    return [t for t in re.findall(r"[A-Za-z][A-Za-z'’-]*", v) if len(t) > 1]


def _is_common(tok: str, vocab: set) -> bool:
    lt = tok.lower().strip("'’")
    return lt in COMMON_WORDS or lt in vocab


def _looks_like_common_phrase(v: str, vocab: set) -> bool:
    """True if the span is made of ordinary words ("Vendor Tier", "T. Please"), not a name."""
    toks = _name_tokens(v)
    if not toks:
        return True
    common = sum(_is_common(t, vocab) for t in toks)
    if common == len(toks):
        return True
    # every token is a common word or an OCR near-miss of one ("Sotup Rigk" = "Setup Risk")
    if len(toks) >= 2 and all(_is_common(t, vocab) or _near_common(t, vocab, 4) for t in toks):
        return True
    return len(toks) >= 3 and common * 2 > len(toks)


def _near_common(tok: str, vocab: set, min_len: int = 5) -> bool:
    """OCR-damaged common words: 'Acommon' (A + common), 'Chiet' (chief), 'Isyour' (is + your)."""
    lt = tok.lower()
    stops = _stop_words()
    if any(lt[:i] in stops and lt[i:] in stops for i in range(2, len(lt) - 1)):
        return True  # two function words run together by OCR ("Isyour")
    if lt[:1] in ("a", "i") and (lt[1:] in vocab or lt[1:] in COMMON_WORDS) and len(lt) > 4:
        return True
    if len(lt) < min_len:
        return False
    for w in vocab | COMMON_WORDS:
        if abs(len(w) - len(lt)) <= 1 and (w[:2] == lt[:2] or w[-2:] == lt[-2:]) and _edit1(lt, w):
            return True
    return any(w.startswith(lt[:-1]) and len(w) >= len(lt) + 2 for w in vocab if w[:3] == lt[:3])


def _bad_person(v: str, vocab: set = frozenset()) -> bool:
    t = v.strip(" .,;:|()[]#*-")
    if not t or t.lower() in PERSON_DENYLIST:
        return True
    if re.search(r"[_\\/=<>{}]", t):          # markup / identifiers, not names
        return True
    if any(re.search(r"[a-z][A-Z]", w) and not w.startswith(("Mc", "Mac")) for w in t.split()):
        return True                            # CamelCase product names (OneTrust, ServPro)
    if _looks_like_common_phrase(t, vocab):
        return True
    if any(c.isdigit() for c in t) or "@" in t or "|" in t or "\n\n" in v:
        return True
    letters = [c for c in t if c.isalpha()]
    if len(letters) < 3:
        return True
    if t.isupper() and len(t) <= 5:      # acronyms (TPRM, KRI...)
        return True
    if not t[0].isupper():
        return True
    return False


ROLE_TAIL = re.compile(r"(?:[\s,]+(?:Dir|Dir\.|Director|VP|SVP|EVP|CEO|CFO|COO|CIO|CRO|CPO|DPO|CISO|Chief|Head|"
                       r"Officer|Counsel|General|Group|Manager|Lead|Secretary|Chair|NED))+\.?$")


def clean(findings: list, text: str, non_pii: list) -> list:
    """Drop invalid/denylisted spans and anything inside a known non-PII business reference."""
    vocab = lowercase_vocab(text)
    trusted = {t for f in findings if f.entity_type == "PERSON" and len(_name_tokens(f.value)) >= 2
               for t in _name_tokens(f.value)}
    out = []
    for f in findings:
        if f.end <= f.start:
            continue
        if any(s <= f.start and f.end <= e for s, e in non_pii):
            continue
        if f.entity_type == "PERSON" and _bad_person(f.value, vocab):
            continue
        if f.entity_type == "PERSON":
            # spaCy sometimes runs a name into the next table cell / line: cut at the break
            cut = re.search(r"\s*(?:\n|\|)", f.value)
            if cut:
                f = Finding(**{**f.__dict__, "end": f.start + cut.start(), "value": f.value[:cut.start()]})
            tail = ROLE_TAIL.search(f.value)  # "S. Aarnio Dir" -> "S. Aarnio"
            if tail and tail.start() > 0:
                f = Finding(**{**f.__dict__, "end": f.start + tail.start(), "value": f.value[:tail.start()]})
            if _bad_person(f.value, vocab):
                continue
            toks = _name_tokens(f.value)
            if len(toks) == 1 and toks[0] not in trusted and (len(toks[0]) <= 3 or _near_common(toks[0], vocab)):
                continue
        out.append(f)
    return out


def resolve(findings: list) -> list:
    """Cluster overlapping spans; keep the union span (never leak a partial value) and the best type."""
    fs = sorted(findings, key=lambda f: (f.start, -f.end))
    clusters, cur, cur_end = [], [], -1
    for f in fs:
        if cur and f.start < cur_end:
            cur.append(f)
            cur_end = max(cur_end, f.end)
        else:
            if cur:
                clusters.append(cur)
            cur, cur_end = [f], f.end
    if cur:
        clusters.append(cur)
    out = []
    for c in clusters:
        best = max(c, key=lambda f: (TYPE_PRIORITY.get(f.entity_type, 0), METHOD_PRIORITY.get(f.method, 0),
                                     f.end - f.start, f.score))
        s, e = min(f.start for f in c), max(f.end for f in c)
        methods = sorted({f.method for f in c})
        recs = sorted({f.recognizer for f in c})
        out.append(Finding(best.entity_type, s, e, "", max(f.score for f in c), "+".join(methods), ",".join(recs)))
    return out


_TOKEN = re.compile(r"[A-Z][a-zA-Z'’-]{3,}")
_FIRST = r"[A-Z][a-z]+(?:-[A-Z][a-z]+)?"
_NOT_FIRST = {"The", "Led", "By", "Dear", "From", "To", "And", "Or", "Of", "For", "With", "Chair", "Owner",
              "Author", "Contact", "Mobile", "Email", "Name", "Role", "Attested", "Challenged", "Approved"}


def _plausible_first(w: str, pos: int, text: str, known: set, vocab: set) -> bool:
    """Should the capitalised word before a known surname be redacted as a first name?"""
    if w in known:
        return True
    if w in _NOT_FIRST or _is_common(w, vocab) or w.lower() in _stop_words():
        return False
    before = text[:pos].rstrip(" \t")
    return not before.endswith((".", "!", "?"))  # sentence-initial capital is weak evidence


@lru_cache(maxsize=1)
def _stop_words() -> frozenset:
    try:
        from spacy.lang.en.stop_words import STOP_WORDS
        return frozenset(STOP_WORDS) | {"later", "earlier", "then", "today", "yesterday", "meanwhile"}
    except Exception:
        return frozenset({"later", "then", "the", "and"})


def _email_tokens(findings: list, text: str) -> set:
    """Name parts from e-mail local parts: 'p.raghunathan@' -> {'Raghunathan'}."""
    out = set()
    for f in findings:
        if f.entity_type != "EMAIL":
            continue
        local = text[f.start:f.end].split("@")[0]
        for part in re.split(r"[._-]+", local):
            if part.isalpha() and len(part) >= 4:
                out.add(part[0].upper() + part[1:].lower())
    return out


def propagate_names(findings: list, text: str) -> list:
    """Find further mentions of people already evidenced in the document.

    Seeds: tokens of detected PERSON spans ("Rafael E. Duarte" -> "Rafael", "Duarte") and name parts of
    detected e-mail addresses. A seed is used only if it is capitalised every time it occurs in prose
    (proper-noun signal), so common words are not swept up. A matched surname is extended to an
    immediately preceding capitalised first name ("Priyanka Raghunathan").
    """
    # seed only from multi-token names: a lone NER token ("Incontrollec") is too weak to spread
    names = {text[f.start:f.end] for f in findings
             if f.entity_type == "PERSON" and len(_name_tokens(text[f.start:f.end])) >= 2}
    # lowercase mentions inside e-mail addresses ("h.park@...") are not evidence of a common word
    prose = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " ", text)
    seeds = {t for n in names for t in _TOKEN.findall(n)} | _email_tokens(findings, text)
    tokens = set()
    for t in seeds:
        if t.endswith(("'s", "’s")):
            t = t[:-2]
        if t.lower() in PERSON_DENYLIST or len(t) < 4 or t.isupper() or _is_common(t, set()):
            continue
        if re.search(rf"\b{re.escape(t.lower())}\b", prose):
            continue
        if not re.search(rf"\b{re.escape(t)}\b", prose):
            continue
        tokens.add(t)
    covered = [(f.start, f.end) for f in findings]
    out = []
    # OCR variants of known name tokens ("Burnev" for "Burney"): edit distance 1, capitalised, not a common word
    vocab = lowercase_vocab(text)
    variants = set()
    for w in set(re.findall(r"\b[A-Z][a-z]{4,}\b", text)) - tokens:
        if _is_common(w, vocab):
            continue
        if any(len(t) >= 5 and _edit1(w, t) for t in tokens):
            variants.add(w)
    for t in sorted(tokens | variants):
        for m in re.finditer(rf"(?:\b({_FIRST})[ \t]+)?\b{re.escape(t)}\b", text):
            s = m.start()
            first = m.group(1)
            if first and not _plausible_first(first, m.start(), text, tokens, vocab):
                s = m.end() - len(t)
            if any(a < m.end() and s < b for a, b in covered):
                s = m.end() - len(t)  # fall back to the surname token alone
                if any(a < m.end() and s < b for a, b in covered):
                    continue
            out.append(Finding("PERSON", s, m.end(), text[s:m.end()], 0.7 if t in tokens else 0.6, "propagation",
                               "known_name_token" if t in tokens else "ocr_name_variant"))
            covered.append((s, m.end()))
    return out


def _edit1(a: str, b: str) -> bool:
    """True if a and b differ by exactly one substitution, insertion or deletion."""
    if a == b or abs(len(a) - len(b)) > 1:
        return False
    if len(a) == len(b):
        return sum(x != y for x, y in zip(a, b)) == 1
    if len(a) > len(b):
        a, b = b, a
    i = 0
    while i < len(a) and a[i] == b[i]:
        i += 1
    return a[i:] == b[i + 1:]


def join_adjacent_persons(findings: list, text: str) -> list:
    """'[PERSON_1] [PERSON_2]' for one name is confusing: merge PERSON spans separated only by spaces/hyphen."""
    fs = sorted(findings, key=lambda f: f.start)
    out = []
    for f in fs:
        if out and out[-1].entity_type == "PERSON" and f.entity_type == "PERSON":
            gap = text[out[-1].end:f.start]
            if len(gap) <= 2 and gap.strip(" -") == "" and "\n" not in gap:
                p = out[-1]
                out[-1] = Finding("PERSON", p.start, f.end, "", max(p.score, f.score),
                                  "+".join(sorted(set(p.method.split("+")) | set(f.method.split("+")))),
                                  ",".join(sorted(set(p.recognizer.split(",")) | set(f.recognizer.split(",")))))
                continue
        out.append(f)
    return out


_INITIALS_BEFORE = re.compile(r"(?:^|(?<=[\s|(/]))([A-Z]{1,3}\.?)[ \t]+$")


def extend_initials(findings: list, text: str) -> list:
    """Include leading initials Presidio leaves out: 'KC Burney', 'DK Koran'."""
    out = []
    for f in findings:
        if f.entity_type == "PERSON":
            m = _INITIALS_BEFORE.search(text, max(0, f.start - 6), f.start)
            if m and m.group(1).rstrip(".").lower() not in PERSON_DENYLIST | {"vp", "svp", "evp", "id", "no"}:
                f = Finding(**{**f.__dict__, "start": m.start(1), "value": text[m.start(1):f.end]})
        out.append(f)
    return out

In [ ]:
%%writefile cadence_pii/detect/presidio_det.py
"""Microsoft Presidio layer (spaCy en_core_web_lg NER + Presidio's built-in recognisers).

Runs per block so tables and image text keep local context; offsets are shifted back to the
document coordinate system. Only entity types in config.PRESIDIO_MAP are kept.
"""
from __future__ import annotations

from functools import lru_cache

from ..config import PRESIDIO_MAP
from ..models import Finding


@lru_cache(maxsize=1)
def get_analyzer():
    from presidio_analyzer import AnalyzerEngine
    from presidio_analyzer.nlp_engine import NlpEngineProvider

    provider = NlpEngineProvider(nlp_configuration={
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}],
    })
    return AnalyzerEngine(nlp_engine=provider.create_engine(), supported_languages=["en"])


@lru_cache(maxsize=1)
def supported_entities() -> list:
    """Only request entities this Presidio build has recognisers for (avoids silent no-ops/warnings)."""
    have = set(get_analyzer().get_supported_entities(language="en"))
    return [e for e in PRESIDIO_MAP if e in have]


def detect_presidio(text: str, offset: int = 0, threshold: float = 0.5) -> list:
    if not text.strip():
        return []
    res = get_analyzer().analyze(text=text, language="en", entities=supported_entities(),
                                 score_threshold=threshold)
    out = []
    for r in res:
        etype = PRESIDIO_MAP.get(r.entity_type)
        if not etype:
            continue
        rec = (r.recognition_metadata or {}).get("recognizer_name", r.entity_type)
        out.append(Finding(etype, r.start + offset, r.end + offset, text[r.start:r.end], round(r.score, 3),
                           "presidio", rec))
    return out

In [ ]:
%%writefile cadence_pii/detect/regex_rules.py
"""Deterministic pattern layer. High precision for structured identifiers; context rules for
identifiers that are only meaningful next to a keyword ("card ending 2218", "DOB 14 July 1991").

Each rule: (entity_type, name, compiled_regex, score, group). `group` selects the sensitive part
of the match (0 = whole match), so the keyword itself is left visible for context.
"""
from __future__ import annotations

import re

from ..models import Finding

MONTH = r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
DATE = (rf"(?:\d{{1,2}}(?:st|nd|rd|th)?\s+{MONTH}\.?,?\s+\d{{4}}"
        rf"|{MONTH}\.?\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{4}}"
        r"|\d{1,2}[/.-]\d{1,2}[/.-](?:\d{4}|\d{2})"
        r"|\d{4}-\d{2}-\d{2})")
STREET_SUFFIX = (r"(?:Street|St\.?|Avenue|Ave\.?|Road|Rd\.?|Lane|Ln\.?|Drive|Dr\.?|Boulevard|Blvd\.?|Crescent|"
                 r"Court|Ct\.?|Place|Pl\.?|Way|Terrace|Close|Square|Sq\.?|Parkway|Pkwy\.?|Highway|Hwy\.?)")
NAME_TOKEN = r"[A-Z][a-z]+(?:['’-][A-Z]?[a-z]+)*"

_F = re.IGNORECASE

RULES = [
    # --- contact details -------------------------------------------------------------
    ("EMAIL", "email",
     re.compile(r"(?<![\w.+-])[A-Za-z0-9][A-Za-z0-9._%+-]*\s?@\s?[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}\b"), 0.99, 0),
    # North American with country code, incl. the short fictional form "+1 (555) 0114"
    ("PHONE", "phone_nanp_intl",
     re.compile(r"(?<![\w+])\+\s?1[\s.]*(?:\(\d{3}\)|\d{3})[\s.-]*(?:\d{3}[\s.-]?\d{4}|\d{4})(?![\d/])"), 0.95, 0),
    # Other international numbers: +CC then 2-5 digit groups; a group may not run into a date ("31/08/2026")
    ("PHONE", "phone_intl",
     re.compile(r"(?<![\w+])\+\s?(?!1\b)\d{2,3}(?:[\s.\-–—]{1,3}\(?\d{1,10}\)?(?![/\d])){1,5}"), 0.9, 0),
    ("PHONE", "phone_us_national",
     re.compile(r"(?<![\d-])(?:\(\d{3}\)\s?|\d{3}[-.])\d{3}[-.\s]\d{4}(?![\d-])"), 0.85, 0),
    ("PHONE", "phone_context",
     re.compile(r"\b(?:tel|phone|mobile|mob|cell|ext(?:ension)?)\b\.?[:\s]*((?:\+?\d[\d\s().-]{6,}\d))", _F), 0.8, 1),
    # --- identifiers -----------------------------------------------------------------
    ("EMPLOYEE_ID", "employee_id",
     re.compile(r"\b(?:EMP|DIR|STF|EMPL)[.,]?\s?[-–]?\s?\d{4,6}\b"), 0.95, 0),  # tolerates OCR "EMP.-38104"
    # vendor personnel IDs: MER-IN-4471, "MER- PH-2214"
    ("EMPLOYEE_ID", "vendor_personnel_id",
     re.compile(r"\b[A-Z]{3}-\s?[A-Z]{2}-\d{4}\b"), 0.85, 0),
    ("EMPLOYEE_ID", "employee_id_context",
     re.compile(r"\b(?:employee|staff|personnel)\s*(?:id|no|number)\b\.?[:\s#]*([A-Z0-9][A-Z0-9-]{3,12})\b", _F), 0.8, 1),
    ("US_SSN", "ssn",
     re.compile(r"(?<![\d-])\d{3}-\d{2}-\d{4}(?![\d-])"), 0.9, 0),
    ("US_SSN", "ssn_context_partial",
     re.compile(r"\b(?:SSN|social security(?: number)?)\b[^\d\n]{0,25}\(?(\d{4})\)?(?![\d-])", _F), 0.8, 1),
    ("PASSPORT", "passport_context",
     re.compile(r"\bpassport(?:\s*(?:no\.?|number|#))?[:\s#]*([A-Z]{1,2}\d{6,9})\b", _F), 0.95, 1),
    ("PASSPORT", "passport_shape",
     re.compile(r"\b[A-Z]{2}\d{7}\b"), 0.6, 0),
    ("NATIONAL_ID", "pesel_context",
     re.compile(r"\bPESEL\b[:\s#]*([\dxX*]{6,11})", _F), 0.95, 1),
    ("NATIONAL_ID", "uk_nino",
     re.compile(r"\b(?!BG|GB|NK|KN|TN|NT|ZZ)[A-CEGHJ-PR-TW-Z]{2}\s?\d{2}\s?\d{2}\s?\d{2}\s?[A-D]\b"), 0.8, 0),
    ("NATIONAL_ID", "national_id_context",
     re.compile(r"\b(?:national (?:id|identifier|identity number)|aadhaar|NINO|NI number)\b[:\s#]*([A-Z0-9][A-Z0-9 -]{5,16}[A-Z0-9])", _F), 0.85, 1),
    ("TAX_ID", "in_pan",
     re.compile(r"\b[A-Z]{3}[ABCFGHLJPT][A-Z]\d{4}[A-Z]\b"), 0.9, 0),
    ("TAX_ID", "tax_context",
     re.compile(r"\b(?:PAN|TIN|EIN|tax (?:id|identifier|number))\b[:\s#]*([A-Z0-9][A-Z0-9-]{6,14})\b", _F), 0.85, 1),
    ("FINANCIAL_ACCOUNT", "account_fragment",
     re.compile(r"\b(?:card|account|acct)\b(?:\s+(?:number|no\.?))?\s+(?:ending(?:\s+in)?|no\.?|number|#)\s*[:#]?\s*(\d{4,19})\b", _F), 0.9, 1),
    ("FINANCIAL_ACCOUNT", "iban",
     re.compile(r"\b[A-Z]{2}\d{2}(?:\s?[A-Z0-9]{4}){3,7}(?:\s?[A-Z0-9]{1,3})?\b"), 0.7, 0),
    # --- dates of birth (context-bound; other dates are not personal by default) -----
    ("DATE_OF_BIRTH", "dob_context",
     re.compile(rf"\b(?:DOB|D\.O\.B\.?|date of birth|birth\s?date|born(?:\s+on)?)\b[\s:,]*(?:is\s+|of\s+)?({DATE})", _F), 0.95, 1),
    # --- addresses -------------------------------------------------------------------
    ("ADDRESS", "street_address",
     re.compile(rf"\b\d{{1,5}}[ \t]+(?:{NAME_TOKEN}[ \t]){{1,3}}{STREET_SUFFIX}(?![A-Za-z])"
                r"(?:,?\s*(?:Apartment|Apt\.?|Suite|Ste\.?|Unit|Floor|Fl\.?)\s*[\w-]+)?"
                r"(?:,\s*[A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+){0,2})?"
                r"(?:,\s*[A-Z]{2}\s+\d{5}(?:-\d{4})?)?"), 0.85, 0),
    # --- names with strong surface cues (NER does the rest) ----------------------------
    # middle initial, tolerating OCR noise: "Caleb A, Whitfield", "Yusuf ©. Balogun", "lfeoma C. Adeyemi"
    ("PERSON", "name_middle_initial",
     re.compile(rf"(?:\b{NAME_TOKEN}|(?<![\w])l[a-z]{{3,}})(?:-{NAME_TOKEN})?\s[A-Z©][.,]\s{NAME_TOKEN}(?:-{NAME_TOKEN})?\b"), 0.9, 0),
    # initials + surname: "R. M. Achterberg", "M, T, Achebe", "L.-E. Brannstrom"
    ("PERSON", "name_initials_surname",
     re.compile(rf"(?<![\w.])(?:[A-Z][.,]-?\s?){{1,2}}{NAME_TOKEN}(?:-{NAME_TOKEN})?\b"), 0.6, 0),
    ("PERSON", "name_with_title",
     re.compile(rf"\b(?:Mr|Mrs|Ms|Miss|Mx|Dr|Prof)\.?\s+{NAME_TOKEN}(?:[\s-]{NAME_TOKEN}){{0,2}}"), 0.85, 0),
]

# Business identifiers that look like PII patterns but are not personal (document refs, tickets).
NON_PII_PATTERNS = [
    re.compile(r"\b(?:GRP|ISS|INC|LOS|RSK|KRI|KCI|OP|POL)-[\w-]+\b"),
]


def detect_regex(text: str, offset: int = 0) -> list:
    out = []
    for etype, name, rx, score, grp in RULES:
        for m in rx.finditer(text):
            s, e = m.span(grp)
            if s < 0 or e <= s:
                continue
            val = text[s:e]
            # trim trailing punctuation that patterns can drag in
            while val and val[-1] in ".,;:)" and not (val.endswith(")") and "(" in val):
                val = val[:-1]
                e -= 1
            if not val.strip():
                continue
            digits = sum(c.isdigit() for c in val)
            if etype == "PHONE" and not (7 <= digits <= 15):
                continue
            # keyword-anchored identifier rules must capture something identifier-like
            if name.endswith("_context") and etype not in ("DATE_OF_BIRTH", "PHONE") and digits < 2:
                continue
            out.append(Finding(etype, s + offset, e + offset, val, score, "regex", name))
    return out


def non_pii_spans(text: str) -> list:
    return [m.span() for rx in NON_PII_PATTERNS for m in rx.finditer(text)]

In [ ]:
%%writefile cadence_pii/detect/run.py
"""Run all enabled detection layers over a Document and return merged, located findings."""
from __future__ import annotations

from ..config import DEFAULT, Settings
from ..models import Document, Finding
from . import merge
from .regex_rules import detect_regex, non_pii_spans


class DetectorError(RuntimeError):
    """Raised when a detection layer fails; the pipeline must fail closed (status ERROR)."""


def detect_text(text: str, settings: Settings = DEFAULT, presidio_threshold=None, segments=None) -> list:
    """Raw findings from every enabled layer. `segments` = [(start, end)] to scan separately."""
    segments = segments or [(0, len(text))]
    thr = settings.presidio_threshold if presidio_threshold is None else presidio_threshold
    found = []
    for s, e in segments:
        seg = text[s:e]
        found += detect_regex(seg, s)
        if settings.use_presidio:
            try:
                from .presidio_det import detect_presidio
                found += detect_presidio(seg, s, thr)
            except Exception as ex:
                raise DetectorError(f"Presidio failed: {type(ex).__name__}: {ex}") from ex
        if settings.use_gliner:
            try:
                from .gliner_det import detect_gliner
                found += detect_gliner(seg, s, settings.gliner_model, settings.gliner_threshold)
            except Exception as ex:
                raise DetectorError(f"GLiNER failed: {type(ex).__name__}: {ex}") from ex
    return found


def finalize(raw: list, text: str, doc: Document = None) -> list:
    """Clean -> propagate known names -> resolve overlaps -> attach values and source locators."""
    fs = merge.clean(raw, text, non_pii_spans(text))
    fs += merge.propagate_names(fs, text)
    fs = merge.join_adjacent_persons(merge.resolve(merge.extend_initials(fs, text)), text)
    for f in fs:
        f.value = text[f.start:f.end]
        if doc is not None:
            b = doc.block_at(f.start)
            if b is not None:
                f.block_id, f.page = b.block_id, b.page
                # add the line within the block so table rows can be found quickly
                bstart = next(s for s, _e, bid in doc.spans if bid == b.block_id)
                line = text[bstart:f.start].count("\n") + 1
                f.locator = f"{b.locator} / line {line}"
    return fs


def detect_document(doc: Document, settings: Settings = DEFAULT) -> list:
    raw = detect_text(doc.text, settings, segments=[(s, e) for s, e, _ in doc.spans])
    return finalize(raw, doc.text, doc)

In [ ]:
%%writefile cadence_pii/evaluate.py
"""Evaluation against the manually annotated gold set + structure-retention measurement.

Metrics (computed per region, aggregated per file):
* redaction recall  - share of gold PII spans fully covered by some redaction (privacy metric)
* typed recall      - share fully covered by a redaction of the same category
* precision         - share of predicted spans that overlap a gold (or optional) span
* context retention - share of non-PII characters in the region that survive redaction
Structure retention (see structure_docx_pptx / structure_pdf) is reported separately.
"""
from __future__ import annotations

import difflib
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

from .config import CATEGORY
from .models import mask_value


def region_segments(doc, sel: dict) -> list:
    out = []
    for (s, e, bid) in doc.spans:
        b = next(x for x in doc.blocks if x.block_id == bid)
        if "page" in sel and b.page != sel["page"]:
            continue
        if "locator_contains" in sel and sel["locator_contains"] not in b.locator:
            continue
        if b.kind in sel.get("exclude_kinds", []):
            continue
        out.append((s, e))
    return out


def _find_all(text, segs, value):
    spans = []
    for s, e in segs:
        i = text.find(value, s, e)
        while i >= 0 and i + len(value) <= e:
            spans.append((i, i + len(value)))
            i = text.find(value, i + 1, e)
    return spans


def _drop_contained(spans):
    """Gold spans contained in a longer gold span ("Rafael" inside "Rafael E. Duarte") are one mention."""
    out = []
    for sp in spans:
        if not any(o != sp and o[0] <= sp[0] and sp[1] <= o[1] and (o[1] - o[0]) > (sp[1] - sp[0])
                   for o in spans):
            out.append(sp)
    return sorted(set(out), key=lambda x: (x[0], x[1]))


def _covered(text, span, preds, same_cat=None):
    """True if every non-space character of the gold span lies inside a predicted span."""
    s, e = span
    pos = [i for i in range(s, e) if not text[i].isspace() and text[i] not in "|"]
    for i in pos:
        if not any(p.start <= i < p.end and (same_cat is None or CATEGORY.get(p.entity_type) == same_cat)
                   for p in preds):
            return False
    return True


def evaluate_file(result, gold: dict) -> dict:
    doc = result.document
    text = doc.text
    agg = Counter()
    per_type = defaultdict(Counter)
    missed, fps, gold_errors = [], [], []
    region_rows = []
    for reg in gold["regions"]:
        segs = region_segments(doc, reg["selector"])
        if not segs:
            gold_errors.append(f"{reg['id']}: selector matched no blocks")
            continue
        gspans, gtype = [], {}
        for etype, val in reg["items"]:
            found = _find_all(text, segs, val)
            if not found:
                gold_errors.append(f"{reg['id']}: gold value not found in text ({etype}, {mask_value(val)})")
            for sp in found:
                gspans.append(sp)
                gtype.setdefault(sp, etype)
        gspans = _drop_contained(gspans)
        ospans = [sp for etype, val in reg.get("optional", []) for sp in _find_all(text, segs, val)]
        preds = [f for f in result.findings if f.start >= 0 and any(f.start < e and s < f.end for s, e in segs)]
        r = Counter()
        for sp in gspans:
            t = gtype[sp]
            ok = _covered(text, sp, preds)
            ok_typed = ok and _covered(text, sp, preds, CATEGORY.get(t))
            r["gold"] += 1
            r["redacted"] += ok
            r["typed"] += ok_typed
            per_type[t]["gold"] += 1
            per_type[t]["redacted"] += ok
            per_type[t]["typed"] += ok_typed
            if not ok:
                touched = any(p.start < sp[1] and sp[0] < p.end for p in preds)
                r["partial"] += touched
                missed.append({"region": reg["id"], "type": t, "kind": "partial" if touched else "missed",
                               "value_masked": mask_value(text[sp[0]:sp[1]]), "_raw": text[sp[0]:sp[1]]})
        for p in preds:
            hit = any(p.start < e and s < p.end for s, e in gspans + ospans)
            r["pred"] += 1
            r["tp_pred"] += hit
            per_type[p.entity_type]["pred"] += 1
            per_type[p.entity_type]["tp_pred"] += hit
            if not hit:
                fps.append({"region": reg["id"], "type": p.entity_type, "method": p.method,
                            "value_masked": mask_value(p.value), "_raw": p.value})
        # context retention: characters outside gold/optional PII that were nevertheless redacted
        region_chars = sum(e - s for s, e in segs)
        pii_chars = set()
        for s, e in gspans + ospans:
            pii_chars.update(range(s, e))
        over = set()
        for p in preds:
            over.update(i for i in range(max(p.start, 0), p.end) if i not in pii_chars
                        and any(s <= i < e for s, e in segs) and not text[i].isspace())
        nonpii = sum(1 for s, e in segs for i in range(s, e) if i not in pii_chars and not text[i].isspace())
        r["nonpii_chars"] += nonpii
        r["over_chars"] += len(over)
        agg.update(r)
        region_rows.append({"region": reg["id"], **r, "chars": region_chars})
    def ratio(a, b):
        return round(a / b, 4) if b else None
    recall = ratio(agg["redacted"], agg["gold"])
    precision = ratio(agg["tp_pred"], agg["pred"])
    summary = {
        "file": gold["file"], "regions": len(region_rows), "gold_spans": agg["gold"], "predicted_spans": agg["pred"],
        "redaction_recall": recall, "typed_recall": ratio(agg["typed"], agg["gold"]),
        "precision": precision, "f1": round(2 * precision * recall / (precision + recall), 4)
        if precision is not None and recall is not None and precision + recall else None,
        "context_retention": ratio(agg["nonpii_chars"] - agg["over_chars"], agg["nonpii_chars"]),
        "not_fully_redacted": len(missed), "of_which_partial": agg["partial"],
        "entirely_missed": len(missed) - agg["partial"], "false_positives": len(fps),
        "per_type": {t: {"gold": c["gold"], "redaction_recall": ratio(c["redacted"], c["gold"]),
                         "typed_recall": ratio(c["typed"], c["gold"]), "predicted": c["pred"],
                         "precision": ratio(c["tp_pred"], c["pred"])} for t, c in sorted(per_type.items())},
    }
    return {"summary": summary, "regions": region_rows, "missed": missed, "false_positives": fps,
            "gold_errors": gold_errors}


# ----------------------------------------------------------------------------- structure retention
def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s.lower())).strip()


def _lis_len(seq):
    import bisect
    tails = []
    for x in seq:
        i = bisect.bisect_left(tails, x)
        tails[i:i + 1] = [x]
    return len(tails)


def structure_docx_pptx(path, extracted_doc) -> dict:
    """Reference = python-docx / python-pptx parse of the OOXML (no OCR). For every reference element
    (heading, paragraph, list item, table cell) check: text retained, element type retained, and order
    retained (longest increasing subsequence of matched positions). Score = elements satisfying all three."""
    from .extract.native_ext import extract_native
    ref = extract_native(path, ocr=False)
    ext_text = _norm(extracted_doc.text)
    kinds_by_text = defaultdict(set)
    table_text = _norm(" ".join(b.text for b in extracted_doc.blocks if b.kind == "table"))
    for b in extracted_doc.blocks:
        kinds_by_text[_norm(b.text)].add(b.kind)
    elements = []
    for b in ref.blocks:
        if b.kind == "table":
            rows = b.text.split("\n")
            for row in rows[:1] + rows[2:]:  # skip the markdown separator line
                for cell in row.strip("|").split("|"):
                    if _norm(cell):
                        elements.append(("table_cell", _norm(cell)))
        elif b.kind in ("heading", "paragraph", "list_item", "notes"):
            elements.append((b.kind, _norm(b.text)))
    found, typed, positions, cursor = 0, 0, [], 0
    per_kind = defaultdict(Counter)
    for kind, t in elements:
        if not t:
            continue
        per_kind[kind]["total"] += 1
        i = ext_text.find(t, cursor)
        if i < 0:
            i = ext_text.find(t)
        if i < 0:
            continue
        found += 1
        per_kind[kind]["text"] += 1
        if len(t) >= 4:  # 1-3 char cells ("A", "R" in RACI tables) are ambiguous for ordering
            positions.append(i)
            cursor = i
        if kind == "table_cell":
            ok = t in table_text
        elif kind == "heading":
            ok = "heading" in kinds_by_text.get(t, set()) or any(t == k for k, v in kinds_by_text.items() if "heading" in v)
        elif kind == "list_item":
            ok = "list_item" in kinds_by_text.get(t, set()) or "paragraph" in kinds_by_text.get(t, set())
        else:
            ok = True
        typed += ok
        per_kind[kind]["typed"] += ok
    total = sum(c["total"] for c in per_kind.values())
    in_order = _lis_len(positions)
    order_ratio = in_order / len(positions) if positions else 1.0
    return {"reference": "native OOXML parse", "elements": total,
            "text_retained": round(found / total, 4) if total else None,
            "type_retained": round(typed / total, 4) if total else None,
            "order_retained": round(order_ratio, 4),
            # element counts only if its text and its type survived, discounted by the share out of order
            "structure_retention": round(typed / total * order_ratio, 4) if total else None,
            "per_kind": {k: dict(v) for k, v in per_kind.items()}}


def structure_pdf(extracted_doc, gold: dict) -> dict:
    """No native reference exists (image-only PDF). Two proxies:
    1. headings: numbered section headings found in the independent Tesseract reading, matched (fuzzy >= 0.85)
       to Docling heading blocks;
    2. tables / figures on the manually counted sample pages vs Docling's detected tables / figures;
    plus word-level content recall of Docling text against the Tesseract reading."""
    tv = extracted_doc.alt_views.get("tesseract", "")
    ref_heads = [_norm(m.group(0)) for m in re.finditer(r"(?m)^\d{1,2}(?:\.\d{1,2})?\.?\s+[A-Z][^\n]{3,80}$", tv)]
    ref_heads = [h for h in ref_heads if len(h.split()) <= 12]
    got_heads = [_norm(b.text) for b in extracted_doc.blocks if b.kind == "heading"]
    matched = sum(1 for h in ref_heads if difflib.get_close_matches(h, got_heads, 1, 0.85))
    ref = gold.get("structure_reference", {}).get("pages", {})
    t_ref = t_got = f_ref = f_got = 0
    for p, c in ref.items():
        p = int(p)
        t_ref += c["tables"]
        f_ref += c["figures"]
        t_got += min(c["tables"], sum(1 for b in extracted_doc.blocks if b.page == p and b.kind == "table"))
        f_got += min(c["figures"], sum(1 for b in extracted_doc.blocks if b.page == p and b.kind == "image_ocr"))
    words = lambda s: Counter(w for w in re.findall(r"[a-z]{3,}", s.lower()))
    wr, wd = words(tv), words(extracted_doc.text)
    content = sum(min(n, wd[w]) for w, n in wr.items()) / max(1, sum(wr.values()))
    return {"reference": "independent Tesseract reading + manual table/figure count on sample pages",
            "headings_ref": len(ref_heads), "headings_matched": matched,
            "heading_retention": round(matched / len(ref_heads), 4) if ref_heads else None,
            "tables_ref": t_ref, "tables_detected": t_got, "table_retention": round(t_got / t_ref, 4) if t_ref else None,
            "figures_ref": f_ref, "figures_ocrd": f_got, "figure_retention": round(f_got / f_ref, 4) if f_ref else None,
            "word_content_recall_vs_tesseract": round(content, 4)}


def save_sanitized(results: dict, path) -> None:
    """Write evaluation output without raw values (strip the private `_raw` keys)."""
    def strip(o):
        if isinstance(o, dict):
            return {k: strip(v) for k, v in o.items() if not k.startswith("_")}
        if isinstance(o, list):
            return [strip(x) for x in o]
        return o
    Path(path).write_text(json.dumps(strip(results), indent=2), encoding="utf-8")

In [ ]:
%%writefile cadence_pii/extract/__init__.py
"""Extraction layer. `extract()` picks the strategy per file type and never returns silently empty output."""
from .router import extract  # noqa: F401

In [ ]:
%%writefile cadence_pii/extract/docling_ext.py
"""Primary extractor: Docling (layout-aware) for PDF, DOCX and PPTX.

* PDF  -> Docling layout + table-structure models with forced full-page Tesseract OCR (the
          supplied PDF is image-only, so OCR is mandatory). Text inside figures (photos of forms,
          diagrams) is not part of Docling's text output, so each figure crop is OCR'd as well.
* DOCX -> Docling's native OOXML backend. Embedded pictures are OCR'd here with Tesseract,
          because Docling does not OCR images inside Word files.
* PPTX -> Docling's native backend (slides, tables, speaker notes); pictures OCR'd likewise.
Docling's remote services stay disabled (default), so nothing leaves the runtime.
"""
from __future__ import annotations

from functools import lru_cache
from pathlib import Path

from ..config import DEFAULT, Settings
from ..models import Block, Document
from .image_ocr import ocr_image

LABEL_KIND = {
    "title": "heading", "section_header": "heading", "text": "paragraph", "paragraph": "paragraph",
    "list_item": "list_item", "caption": "caption", "footnote": "paragraph", "table": "table",
    "page_header": "furniture", "page_footer": "furniture", "code": "paragraph", "formula": "paragraph",
    "picture": "picture", "chart": "picture", "document_index": "table", "reference": "paragraph",
    "key_value_region": "paragraph", "form": "paragraph", "handwritten_text": "paragraph",
}


def _inside_table(item, d) -> bool:
    """Rich table cells are yielded again as standalone items; they are already in the table text."""
    from docling_core.types.doc import TableItem
    ref = getattr(item, "parent", None)
    while ref is not None:
        node = ref.resolve(d)
        if isinstance(node, TableItem):
            return True
        ref = getattr(node, "parent", None)
    return False


def _native_crop(path: Path, item, d, pad: int = 8):
    """Crop a PDF figure from the page's native-resolution raster (sharper than Docling's render)."""
    try:
        import io

        import pymupdf
        from PIL import Image
        prov = item.prov[0]
        page_no, bb = prov.page_no, prov.bbox
        size = d.pages[page_no].size
        with pymupdf.open(path) as pdf:
            pg = pdf[page_no - 1]
            imgs = pg.get_images(full=True)
            if len(imgs) != 1:
                return None
            img = Image.open(io.BytesIO(pdf.extract_image(imgs[0][0])["image"]))
            # where the raster sits on the page (points)
            r = pg.get_image_rects(imgs[0][0])[0]
        bb = bb.to_top_left_origin(page_height=size.height)
        sx, sy = img.width / r.width, img.height / r.height
        box = (max(0, int((bb.l - r.x0) * sx) - pad), max(0, int((bb.t - r.y0) * sy) - pad),
               min(img.width, int((bb.r - r.x0) * sx) + pad), min(img.height, int((bb.b - r.y0) * sy) + pad))
        if box[2] - box[0] < 20 or box[3] - box[1] < 20:
            return None
        return img.crop(box)
    except Exception:
        return None


@lru_cache(maxsize=2)
def _converter(ocr_lang: str):
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractCliOcrOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    opts = PdfPipelineOptions(
        do_ocr=True,
        do_table_structure=True,
        ocr_options=TesseractCliOcrOptions(force_full_page_ocr=True, lang=[ocr_lang]),
        generate_picture_images=True,  # figure crops are OCR'd separately (Docling omits text inside figures)
        images_scale=2.0,
    )
    return DocumentConverter(format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)})


def extract_docling(path, settings: Settings = DEFAULT) -> Document:
    from docling_core.types.doc import ContentLayer, PictureItem, TableItem

    path = Path(path)
    ext = path.suffix.lower().lstrip(".")
    res = _converter(settings.ocr_lang).convert(str(path), raises_on_error=False)
    if str(res.status).split(".")[-1] not in ("SUCCESS", "PARTIAL_SUCCESS"):
        raise RuntimeError(f"Docling conversion status {res.status}: {[str(e) for e in res.errors][:3]}")
    d = res.document
    doc = Document(filename=path.name, filetype=ext)
    if str(res.status).endswith("PARTIAL_SUCCESS"):
        doc.warn("Docling reported PARTIAL_SUCCESS (some pages/elements not converted)")

    unit = {"pdf": "page", "pptx": "slide"}.get(ext)
    layers = {ContentLayer.BODY, ContentLayer.FURNITURE, ContentLayer.NOTES}
    counters: dict = {}
    section = "start"
    bid = 0
    unsupported_pics = 0
    for item, _lvl in d.iterate_items(included_content_layers=layers):
        if not isinstance(item, PictureItem) and _inside_table(item, d):
            continue
        label = str(getattr(item, "label", "")).split(".")[-1].lower()
        kind = LABEL_KIND.get(label, "paragraph")
        layer = str(getattr(item, "content_layer", "body")).split(".")[-1].lower()
        if layer == "notes":
            kind = "notes"
        elif layer == "furniture" and kind not in ("picture",):
            kind = "furniture"
        page = item.prov[0].page_no if getattr(item, "prov", None) else None
        counters[kind] = counters.get(kind, 0) + 1
        where = f"{unit} {page}" if unit and page else f"section '{section[:60]}'"
        locator = f"{where} / {kind} #{counters[kind]}"

        if isinstance(item, TableItem):
            text = item.export_to_markdown(doc=d)
        elif isinstance(item, PictureItem):
            img = _native_crop(path, item, d) if ext == "pdf" else None
            if img is None:
                img = item.get_image(d)
            if img is None:
                unsupported_pics += 1
                if layer == "furniture":  # header/footer art (logos) - recorded, not blocking
                    doc.warn(f"header/footer picture ({locator}) is vector (EMF/WMF) - not OCR'd", "info")
                else:
                    doc.warn(f"body picture at {locator} is vector (EMF/WMF/SVG) - not OCR'd; check manually")
                continue
            r = ocr_image(img, settings.ocr_lang, settings.ocr_min_block_conf, settings.ocr_min_words_for_conf)
            if not r.text:
                continue
            bid += 1
            doc.blocks.append(Block(bid, locator.replace("picture", "image"), "image_ocr", r.text, "tesseract",
                                    page, r.mean_conf, r.n_words))
            continue
        else:
            text = getattr(item, "text", "") or ""
        if kind == "heading" and text.strip():
            section = text.strip()
        if not text.strip():
            continue
        bid += 1
        doc.blocks.append(Block(bid, locator, kind, text.strip(), "docling", page,
                                None))
    doc.inventory.update({
        "pages_or_slides": len(d.pages) if d.pages else None,
        "tables": len(d.tables),
        "pictures": len(d.pictures),
        "pictures_without_raster": unsupported_pics,
    })
    doc.assemble()
    return doc

In [ ]:
%%writefile cadence_pii/extract/image_ocr.py
"""Local Tesseract OCR helpers (no network). Returns text with line structure and confidence."""
from __future__ import annotations

import io
from dataclasses import dataclass, field

import pytesseract
from PIL import Image, ImageOps


@dataclass
class OcrResult:
    text: str
    mean_conf: float          # mean confidence of real words (0-100); 0 when no words
    n_words: int
    low_conf_regions: list = field(default_factory=list)  # [(block_num, mean_conf, n_words)]


def _prep(img: Image.Image) -> Image.Image:
    img = ImageOps.exif_transpose(img)
    if img.mode not in ("RGB", "L"):
        bg = Image.new("RGB", img.size, "white")
        bg.paste(img.convert("RGBA"), mask=img.convert("RGBA").split()[-1])
        img = bg
    img = img.convert("L")
    # Small UI screenshots OCR much better when upscaled.
    if img.width < 1200:
        f = min(3.0, 1600 / max(img.width, 1))
        img = img.resize((int(img.width * f), int(img.height * f)), Image.LANCZOS)
    return img


def ocr_image(img, lang: str = "eng", region_min_conf: float = 60.0, region_min_words: int = 5) -> OcrResult:
    """OCR a PIL image or raw bytes; rebuild lines/paragraphs from Tesseract's layout tree."""
    if isinstance(img, (bytes, bytearray)):
        img = Image.open(io.BytesIO(img))
    img = _prep(img)
    d = pytesseract.image_to_data(img, lang=lang, config="--psm 3", output_type=pytesseract.Output.DICT)
    lines: dict = {}
    blocks: dict = {}
    confs = []
    for i, w in enumerate(d["text"]):
        w = w.strip()
        c = float(d["conf"][i])
        if not w or c < 0:
            continue
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(w)
        blocks.setdefault(d["block_num"][i], []).append(c)
        confs.append(c)
    out, prev = [], None
    for key in sorted(lines):
        if prev is not None and key[:2] != prev[:2]:
            out.append("")  # blank line between paragraphs
        out.append(" ".join(lines[key]))
        prev = key
    low = [(b, round(sum(c) / len(c), 1), len(c)) for b, c in blocks.items()
           if len(c) >= region_min_words and sum(c) / len(c) < region_min_conf]
    return OcrResult("\n".join(out).strip(), round(sum(confs) / len(confs), 1) if confs else 0.0, len(confs), low)

In [ ]:
%%writefile cadence_pii/extract/native_ext.py
"""Reference / fallback extractor using python-docx and python-pptx directly.

Serves two purposes:
1. Structure-retention baseline: the OOXML parse is ground truth for what text elements exist.
2. Fallback when Docling fails, so a document is never silently skipped.
"""
from __future__ import annotations

import zipfile
from pathlib import Path

from ..config import DEFAULT, Settings
from ..models import Block, Document
from .image_ocr import ocr_image

RASTER = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".tif", ".tiff")


def _table_md(rows) -> str:
    rows = [[c.replace("\n", " ").replace("|", "/").strip() for c in r] for r in rows if any(c.strip() for c in r)]
    if not rows:
        return ""
    w = max(len(r) for r in rows)
    rows = [r + [""] * (w - len(r)) for r in rows]
    lines = ["| " + " | ".join(rows[0]) + " |", "|" + "---|" * w]
    lines += ["| " + " | ".join(r) + " |" for r in rows[1:]]
    return "\n".join(lines)


def _docx(path: Path, ocr: bool, settings: Settings) -> Document:
    import docx
    from docx.table import Table
    from docx.text.paragraph import Paragraph

    d = docx.Document(str(path))
    doc = Document(filename=path.name, filetype="docx")
    bid, section = 0, "start"
    for el in d.element.body.iterchildren():
        tag = el.tag.split("}")[-1]
        if tag == "p":
            p = Paragraph(el, d)
            t = p.text.strip()
            if not t:
                continue
            style = (p.style.name or "").lower() if p.style is not None else ""
            kind = "heading" if style.startswith(("heading", "title")) else (
                "list_item" if "list" in style or el.find(".//{*}numPr") is not None else "paragraph")
            if kind == "heading":
                section = t
            bid += 1
            doc.blocks.append(Block(bid, f"section '{section[:60]}' / {kind}", kind, t, "native"))
        elif tag == "tbl":
            tb = Table(el, d)
            md = _table_md([[c.text for c in r.cells] for r in tb.rows])
            if md:
                bid += 1
                doc.blocks.append(Block(bid, f"section '{section[:60]}' / table", "table", md, "native"))
    for i, s in enumerate(d.sections):
        for part, name in ((s.header, "header"), (s.footer, "footer")):
            t = "\n".join(p.text for p in part.paragraphs if p.text.strip())
            if t:
                bid += 1
                doc.blocks.append(Block(bid, f"section {i+1} {name}", "furniture", t, "native"))
    if ocr:
        with zipfile.ZipFile(path) as z:
            for n in sorted(x for x in z.namelist() if x.startswith("word/media/")):
                if n.lower().endswith(RASTER):
                    r = ocr_image(z.read(n), settings.ocr_lang)
                    if r.text:
                        bid += 1
                        doc.blocks.append(Block(bid, f"embedded image {n.split('/')[-1]}", "image_ocr",
                                                r.text, "tesseract", None, r.mean_conf, r.n_words))
                else:
                    doc.warn(f"embedded image {n.split('/')[-1]} is not a raster format - not OCR'd")
    doc.assemble()
    return doc


def _pptx(path: Path, ocr: bool, settings: Settings) -> Document:
    from pptx import Presentation
    from pptx.enum.shapes import MSO_SHAPE_TYPE

    prs = Presentation(str(path))
    doc = Document(filename=path.name, filetype="pptx")
    bid = 0

    def walk(shapes):
        for sh in shapes:
            if sh.shape_type == MSO_SHAPE_TYPE.GROUP:
                yield from walk(sh.shapes)
            else:
                yield sh

    for sno, slide in enumerate(prs.slides, 1):
        for sh in walk(slide.shapes):
            if sh.has_text_frame and sh.text_frame.text.strip():
                for para in sh.text_frame.paragraphs:
                    t = "".join(r.text for r in para.runs).strip()
                    if t:
                        bid += 1
                        doc.blocks.append(Block(bid, f"slide {sno} / shape '{sh.name}'", "paragraph", t, "native", sno))
            elif getattr(sh, "has_table", False) and sh.has_table:
                md = _table_md([[c.text for c in r.cells] for r in sh.table.rows])
                bid += 1
                doc.blocks.append(Block(bid, f"slide {sno} / table '{sh.name}'", "table", md, "native", sno))
            elif ocr and sh.shape_type == MSO_SHAPE_TYPE.PICTURE:
                try:
                    blob = sh.image.blob
                except Exception:
                    doc.warn(f"slide {sno}: picture '{sh.name}' could not be read - not OCR'd")
                    continue
                r = ocr_image(blob, settings.ocr_lang)
                if r.text:
                    bid += 1
                    doc.blocks.append(Block(bid, f"slide {sno} / image '{sh.name}'", "image_ocr", r.text,
                                            "tesseract", sno, r.mean_conf, r.n_words))
        if slide.has_notes_slide:
            t = slide.notes_slide.notes_text_frame.text.strip() if slide.notes_slide.notes_text_frame else ""
            if t:
                bid += 1
                doc.blocks.append(Block(bid, f"slide {sno} / speaker notes", "notes", t, "native", sno))
    doc.inventory["pages_or_slides"] = len(prs.slides)
    doc.inventory["tables"] = sum(1 for b in doc.blocks if b.kind == "table")
    doc.inventory["pictures"] = sum(1 for b in doc.blocks if b.kind == "image_ocr")
    doc.assemble()
    return doc


def extract_native(path, settings: Settings = DEFAULT, ocr: bool = True) -> Document:
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".docx":
        return _docx(path, ocr, settings)
    if ext == ".pptx":
        return _pptx(path, ocr, settings)
    if ext == ".pdf":
        from .pdf_ocr import extract_pdf_tesseract
        return extract_pdf_tesseract(path, settings)
    raise ValueError(f"unsupported: {ext}")

In [ ]:
%%writefile cadence_pii/extract/pdf_ocr.py
"""Independent full-resolution Tesseract view of a PDF.

Used (a) to measure per-page OCR confidence and find low-confidence regions for human review,
(b) as a second OCR reading that the verification step cross-checks against, and
(c) as a fallback extractor if Docling fails. Pages with a usable text layer use that layer instead.
"""
from __future__ import annotations

import io
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pymupdf
from PIL import Image

from ..config import DEFAULT, Settings
from ..models import Block, Document
from .image_ocr import ocr_image


def _page_image(doc, page) -> Image.Image:
    imgs = page.get_images(full=True)
    if len(imgs) == 1:  # scanned page: use the embedded raster at native resolution
        data = doc.extract_image(imgs[0][0])["image"]
        return Image.open(io.BytesIO(data))
    pix = page.get_pixmap(dpi=200)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def extract_pdf_tesseract(path, settings: Settings = DEFAULT) -> Document:
    path = Path(path)
    out = Document(filename=path.name, filetype="pdf")
    with pymupdf.open(path) as pdf:
        n = pdf.page_count
        native = [pdf[i].get_text("text").strip() for i in range(n)]
        images = [None if len(native[i]) > 50 else _page_image(pdf, pdf[i]) for i in range(n)]

    def work(i):
        if images[i] is None:
            return i, native[i], None
        return i, None, ocr_image(images[i], settings.ocr_lang, settings.ocr_min_block_conf,
                                  settings.ocr_min_words_for_conf)

    with ThreadPoolExecutor(max_workers=max(1, (os.cpu_count() or 2))) as ex:
        results = sorted(ex.map(work, range(n)))
    page_conf = {}
    for i, txt, r in results:
        p = i + 1
        if r is None:
            out.blocks.append(Block(p, f"page {p} / text layer", "paragraph", txt, "native", p))
            continue
        page_conf[p] = r.mean_conf
        out.blocks.append(Block(p, f"page {p} / OCR", "page_ocr", r.text, "tesseract", p, r.mean_conf, r.n_words))
        if not r.text.strip():
            out.warn(f"page {p}: OCR produced no text")
        for (bnum, conf, nw) in r.low_conf_regions:
            out.warn(f"page {p}: low-confidence OCR region (block {bnum}, mean conf {conf}, {nw} words)")
    out.inventory.update({"pages_or_slides": n, "text_layer_pages": sum(1 for t in native if len(t) > 50),
                          "ocr_page_conf": page_conf})
    out.assemble()
    return out

In [ ]:
%%writefile cadence_pii/extract/router.py
"""Choose the extraction strategy per file type and attach secondary views."""
from __future__ import annotations

import time
from pathlib import Path

from ..config import DEFAULT, Settings
from ..models import Document


# Per-format strategy chosen from measurements on the supplied artifacts (docs/file_structure_analysis.md):
# * PDF  - image-only scan: Docling layout/table models + forced Tesseract OCR (+ figure OCR)
# * DOCX - Docling keeps headings/lists/tables (98.5% structure retention); screenshots OCR'd separately
# * PPTX - python-pptx walk of the shape tree keeps each org-chart card (name, title, e-mail, phone)
#          together; Docling regroups them row-wise and loses that association
STRATEGY = {".pdf": ("docling", "native"), ".docx": ("docling", "native"), ".pptx": ("native", "docling")}


def _run(name: str, path: Path, settings: Settings) -> Document:
    if name == "docling":
        from .docling_ext import extract_docling
        return extract_docling(path, settings)
    from .native_ext import extract_native
    return extract_native(path, settings)


def extract(path, settings: Settings = DEFAULT) -> Document:
    """Primary extractor per STRATEGY; on failure fall back to the other one (and say so).

    For PDFs an independent full-resolution Tesseract reading is attached as `alt_views['tesseract']`
    together with per-page confidence and low-confidence-region warnings.
    """
    path = Path(path)
    t0 = time.time()
    primary, fallback = STRATEGY[path.suffix.lower()]
    try:
        doc = _run(primary, path, settings)
        doc.inventory["extractor"] = primary
    except Exception as e:  # fail over, but record it; the gate will require review
        doc = _run(fallback, path, settings)
        doc.inventory["extractor"] = f"{fallback} (fallback)"
        doc.warn(f"{primary} extractor failed ({type(e).__name__}); used {fallback} fallback")
    doc.inventory["extract_secs"] = round(time.time() - t0, 1)

    if path.suffix.lower() == ".pdf" and settings.pdf_dual_ocr and doc.inventory.get("extractor") == "docling":
        from .pdf_ocr import extract_pdf_tesseract
        t1 = time.time()
        tv = extract_pdf_tesseract(path, settings)
        doc.alt_views["tesseract"] = tv.text
        doc.warnings.extend(tv.warnings)
        doc.inventory["ocr_page_conf"] = tv.inventory.get("ocr_page_conf")
        doc.inventory["text_layer_pages"] = tv.inventory.get("text_layer_pages")
        doc.inventory["second_ocr_secs"] = round(time.time() - t1, 1)
        # Docling should not have dropped whole pages: compare page coverage
        docling_pages = {b.page for b in doc.blocks if b.page}
        missing = sorted(set(range(1, (tv.inventory.get("pages_or_slides") or 0) + 1)) - docling_pages)
        if missing:
            doc.warn(f"Docling produced no text for pages {missing}")

    if not doc.text.strip():
        doc.warn("extraction produced no text")
    else:
        _coverage_check(path, doc)
    return doc


def _ooxml_text_chars(path: Path) -> int:
    """Non-whitespace characters in the OOXML text runs (w:t / a:t) - a cheap completeness reference."""
    import re
    import zipfile
    n = 0
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if name.endswith(".xml") and name.startswith(("word/document", "word/header", "word/footer",
                                                          "word/footnotes", "ppt/slides/slide", "ppt/notesSlides/")):
                xml = z.read(name).decode("utf8", "ignore")
                runs = re.findall(r"<(?:w|a):t(?:\s[^>]*)?>([^<]*)</(?:w|a):t>", xml)
                n += sum(1 for t in runs for c in t if not c.isspace())
    return n


def _coverage_check(path: Path, doc: Document, min_ratio: float = 0.8) -> None:
    """Flag extraction that silently dropped content (compares like-for-like non-whitespace characters)."""
    ext = path.suffix.lower()
    if ext in (".docx", ".pptx"):
        ref = _ooxml_text_chars(path)
        got = sum(1 for b in doc.blocks if b.kind != "image_ocr" for c in b.text
                  if not c.isspace() and c not in "|-#")  # ignore markdown table/heading syntax
        doc.inventory["source_text_chars"] = ref
        doc.inventory["extracted_text_chars"] = got
        doc.inventory["text_coverage"] = round(got / ref, 3) if ref else None
        if ref and got < min_ratio * ref:
            doc.warn(f"extracted text covers only {100 * got / ref:.0f}% of the source text runs")
    elif ext == ".pdf":
        pages = doc.inventory.get("pages_or_slides") or 0
        per_page = {}
        for b in doc.blocks:
            if b.page:
                per_page[b.page] = per_page.get(b.page, 0) + len(b.text)
        thin = [p for p in range(1, pages + 1) if per_page.get(p, 0) < 40]
        if thin:
            doc.warn(f"pages with almost no extracted text: {thin}")

In [ ]:
%%writefile cadence_pii/gate.py
"""Release gate. Only CLEARED text can be handed to the downstream AI agent."""
from __future__ import annotations

from datetime import datetime, timezone

from .config import DEFAULT, Settings
from .models import FileResult, Status


class ReleaseBlocked(PermissionError):
    pass


def decide(r: FileResult, settings: Settings = DEFAULT) -> FileResult:
    """Set r.status and r.reasons from extraction warnings, OCR quality and verification results."""
    if r.status == Status.ERROR:
        return r
    reasons = []
    doc = r.document
    for w in doc.warnings:
        if w["level"] == "review":
            reasons.append(("REVIEW", w["msg"]))
    for b in doc.blocks:
        if b.kind == "image_ocr" and b.ocr_conf is not None and b.ocr_words >= settings.ocr_min_words_for_conf \
                and b.ocr_conf < settings.ocr_min_block_conf:
            reasons.append(("REVIEW", f"low OCR confidence ({b.ocr_conf}) for {b.locator}"))
    pii_images = sorted({f.locator.split(" / line")[0] for f in r.findings
                         if (b := doc.block_at(f.start) if f.start >= 0 else None) is not None and b.kind == "image_ocr"})
    if pii_images:
        reasons.append(("REVIEW", f"{len(pii_images)} embedded image(s) contain PII; OCR may miss garbled or partial "
                                  f"names - visually confirm: {', '.join(pii_images[:5])}{' ...' if len(pii_images) > 5 else ''}"))
    v = r.verification.get("final", {})
    if v.get("residual_count", 0) > 0:
        reasons.append(("BLOCK", f"{v['residual_count']} residual PII candidate(s) remain after remediation"))
    if not r.findings and len(doc.text) > 2000:
        reasons.append(("REVIEW", "no PII found in a substantial document - confirm this is expected"))

    if any(k == "BLOCK" for k, _ in reasons):
        r.status = Status.BLOCKED
    elif reasons:
        r.status = Status.REVIEW_REQUIRED
    else:
        r.status = Status.CLEARED
    r.reasons = [f"{k}: {m}" for k, m in reasons]
    return r


def approve(r: FileResult, reviewer: str, note: str) -> FileResult:
    """Human sign-off for REVIEW_REQUIRED files. BLOCKED/ERROR files cannot be approved."""
    if r.status != Status.REVIEW_REQUIRED:
        raise ReleaseBlocked(f"only REVIEW_REQUIRED files can be approved (status is {r.status.value})")
    if not reviewer.strip() or not note.strip():
        raise ValueError("reviewer name and review note are required")
    r.review_log.append({"reviewer": reviewer, "note": note, "at": datetime.now(timezone.utc).isoformat(),
                         "reasons_reviewed": list(r.reasons)})
    r.status = Status.CLEARED
    return r


def get_clean_text(r: FileResult) -> str:
    """The only path to downstream-AI text. Raises unless the file is CLEARED."""
    if r.status != Status.CLEARED:
        raise ReleaseBlocked(f"{r.filename}: status {r.status.value} - not released to downstream AI")
    return r.redacted_text

In [ ]:
%%writefile cadence_pii/ingest.py
"""File validation: extension, size, magic bytes and container integrity."""
import hashlib
import zipfile
from pathlib import Path

from .config import DEFAULT, Settings

OOXML_MAIN_PART = {".docx": "word/document.xml", ".pptx": "ppt/presentation.xml"}


class ValidationError(Exception):
    pass


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def validate(path, settings: Settings = DEFAULT) -> dict:
    """Return basic file facts or raise ValidationError with a clear reason."""
    path = Path(path)
    if not path.exists():
        raise ValidationError("file not found")
    ext = path.suffix.lower()
    if ext not in settings.allowed_ext:
        raise ValidationError(f"unsupported file type '{ext}' (supported: {', '.join(settings.allowed_ext)})")
    size = path.stat().st_size
    if size == 0:
        raise ValidationError("file is empty")
    if size > settings.max_file_mb * 1024 * 1024:
        raise ValidationError(f"file exceeds {settings.max_file_mb} MB limit")
    with open(path, "rb") as f:
        head = f.read(8)
    if ext == ".pdf":
        if not head.startswith(b"%PDF"):
            raise ValidationError("not a valid PDF (missing %PDF header)")
        import pymupdf
        try:
            with pymupdf.open(path) as d:
                if d.page_count == 0:
                    raise ValidationError("PDF has no pages")
                if d.needs_pass:
                    raise ValidationError("PDF is password-protected")
        except ValidationError:
            raise
        except Exception as e:  # corrupt structure
            raise ValidationError(f"PDF could not be opened: {type(e).__name__}")
    else:
        if not head.startswith(b"PK"):
            raise ValidationError(f"not a valid {ext} (not a ZIP/OOXML container)")
        try:
            with zipfile.ZipFile(path) as z:
                bad = z.testzip()
                if bad:
                    raise ValidationError(f"corrupt archive member: {bad}")
                if OOXML_MAIN_PART[ext] not in z.namelist():
                    raise ValidationError(f"{ext} is missing {OOXML_MAIN_PART[ext]}")
        except zipfile.BadZipFile:
            raise ValidationError(f"{ext} archive is corrupt")
    return {"filetype": ext.lstrip("."), "size_bytes": size, "sha256": sha256(path)}

In [ ]:
%%writefile cadence_pii/models.py
"""Data model shared by every pipeline stage."""
from __future__ import annotations

import bisect
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Optional


class Status(str, Enum):
    PROCESSING = "PROCESSING"
    CLEARED = "CLEARED"                  # verified; text may go to the downstream AI
    REVIEW_REQUIRED = "REVIEW_REQUIRED"  # a human must look before release
    BLOCKED = "BLOCKED"                  # residual PII found after redaction; never release
    ERROR = "ERROR"                      # could not be processed (corrupt / unsupported / crash)


@dataclass
class Block:
    """One structural unit of extracted text, in reading order."""
    block_id: int
    locator: str                 # human-readable source location, e.g. "page 21 / table 3"
    kind: str                    # heading | paragraph | list_item | table | caption | image_ocr | page_ocr | notes | furniture
    text: str
    source: str                  # docling | native | tesseract
    page: Optional[int] = None   # page / slide number where known
    ocr_conf: Optional[float] = None  # mean word confidence (0-100) for OCR-derived text
    ocr_words: int = 0


@dataclass
class Finding:
    entity_type: str
    start: int                   # offsets into Document.text (the exact string that was scanned)
    end: int
    value: str                   # raw value: kept in memory only, never written to reports
    score: float
    method: str                  # regex | presidio | gliner | cross_view
    recognizer: str
    block_id: int = -1
    locator: str = ""
    placeholder: str = ""
    red_start: int = -1          # placeholder position in the redacted text
    red_end: int = -1
    page: Optional[int] = None

    def public(self, context: str = "") -> dict:
        d = asdict(self)
        d.pop("value")
        d["value_masked"] = mask_value(self.value)
        d["context_masked"] = context
        return d


@dataclass
class Document:
    """Normalised extraction output: one string plus a block index for traceability."""
    filename: str
    filetype: str
    blocks: list = field(default_factory=list)
    text: str = ""
    spans: list = field(default_factory=list)   # (start, end, block_id) per block, sorted
    warnings: list = field(default_factory=list)  # [{"level": "review"|"info", "msg": str}] -> feed the gate
    alt_views: dict = field(default_factory=dict)  # e.g. {"tesseract": "..."} for cross-checks
    inventory: dict = field(default_factory=dict)

    def warn(self, msg: str, level: str = "review") -> None:
        """Record an extraction issue. 'review' issues stop a file being CLEARED; 'info' is audit-only."""
        self.warnings.append({"level": level, "msg": msg})

    @staticmethod
    def render_block(b: Block) -> str:
        if b.kind == "heading":
            return "## " + b.text
        if b.kind == "list_item":
            return "- " + b.text
        if b.kind == "image_ocr":
            return f"[image text: {b.locator}]\n{b.text}"
        return b.text

    def assemble(self) -> None:
        """Build self.text from blocks, recording each block's character span."""
        parts, spans, pos = [], [], 0
        for b in self.blocks:
            s = self.render_block(b)
            if not s.strip():
                continue
            spans.append((pos, pos + len(s), b.block_id))
            parts.append(s)
            pos += len(s) + 2
        self.text = "\n\n".join(parts)
        self.spans = spans

    def block_at(self, offset: int) -> Optional[Block]:
        starts = [s[0] for s in self.spans]
        i = bisect.bisect_right(starts, offset) - 1
        if i < 0:
            return None
        bid = self.spans[i][2]
        return next((b for b in self.blocks if b.block_id == bid), None)


@dataclass
class FileResult:
    filename: str
    filetype: str = ""
    sha256: str = ""
    size_bytes: int = 0
    status: Status = Status.PROCESSING
    reasons: list = field(default_factory=list)
    document: Optional[Document] = None
    findings: list = field(default_factory=list)
    redacted_text: str = ""
    verification: dict = field(default_factory=dict)
    timings: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)
    review_log: list = field(default_factory=list)  # human sign-offs (who, when, note)


def mask_value(v: str) -> str:
    """Show only shape: first character + length, digits/letters masked. Never reveals the value."""
    v = v.strip()
    if not v:
        return ""
    keep = v[0] if v[0].isalpha() else "*"
    return f"{keep}{'*' * (min(len(v), 12) - 1)} ({len(v)} chars)"

In [ ]:
%%writefile cadence_pii/pipeline.py
"""End-to-end orchestration. One bad file never stops the batch; failures become ERROR results."""
from __future__ import annotations

import sys
import time
import traceback
from pathlib import Path

from . import gate
from .config import DEFAULT, Settings
from .detect.run import detect_document
from .extract import extract
from .ingest import ValidationError, validate
from .models import FileResult, Finding, Status
from .redact import redact
from .verify import verify


def _locate(doc, value: str):
    i = doc.text.find(value)
    return doc.block_at(i) if i >= 0 else None


def run_file(path, settings: Settings = DEFAULT) -> FileResult:
    path = Path(path)
    r = FileResult(filename=path.name)
    t = {}
    try:
        t0 = time.time()
        r.__dict__.update(validate(path, settings))
        t["validate"] = round(time.time() - t0, 2)

        t0 = time.time()
        doc = extract(path, settings)
        r.document = doc
        t["extract"] = round(time.time() - t0, 1)
        if not doc.text.strip():
            r.status = Status.ERROR
            r.errors.append("no text could be extracted")
            return r

        t0 = time.time()
        r.findings = detect_document(doc, settings)
        t["detect"] = round(time.time() - t0, 1)

        t0 = time.time()
        redacted, mapping = redact(doc.text, r.findings)
        t["redact"] = round(time.time() - t0, 2)

        t0 = time.time()
        alt = doc.alt_views.get("tesseract", "")
        v1 = verify(redacted, r.findings, alt, settings)
        r.verification["initial"] = _summ(v1)
        # one remediation pass: redact what verification found, then verify again
        extra = v1["rescan"] + v1["cross_view"] + [Finding("LEAK", f.start, f.end, f.value, 1, f.method, f.recognizer)
                                                    for f in v1["leaks"]]
        if extra:
            from .detect.merge import resolve
            fix = resolve(extra)
            for f in fix:
                f.value = redacted[f.start:f.end]
                if f.entity_type == "LEAK":
                    orig = next((g for g in r.findings if g.value.strip() == f.value.strip()), None)
                    f.entity_type = orig.entity_type if orig else "PERSON"
            redacted, mapping = redact(redacted, fix, mapping)
            deltas = [(f.start, len(f.placeholder) - (f.end - f.start)) for f in fix]
            for f in fix:
                i = doc.text.find(f.value)  # report offsets in the extracted-text coordinate system
                f.start, f.end = (i, i + len(f.value)) if i >= 0 else (-1, -1)
                b = _locate(doc, f.value)
                f.block_id, f.page = (b.block_id, b.page) if b else (-1, None)
                f.locator = (b.locator if b else "unknown") + " (remediation pass)"
                f.method = "verify:" + f.method
            # recompute placeholder positions of the original findings in the final text
            r.findings = _shift(r.findings, deltas) + fix
            v2 = verify(redacted, r.findings, alt, settings)
        else:
            v2 = v1
        r.verification["final"] = _summ(v2)
        r.verification["remediated"] = len(extra)
        r.redacted_text = redacted
        t["verify"] = round(time.time() - t0, 1)
        gate.decide(r, settings)
    except ValidationError as e:
        r.status = Status.ERROR
        r.errors.append(f"validation: {e}")
    except Exception as e:  # fail closed on anything unexpected (detector crash, OCR failure...)
        r.status = Status.ERROR
        r.errors.append(f"{type(e).__name__}: {e}")
        r.errors.append(traceback.format_exc(limit=3))
        r.redacted_text = ""
    finally:
        r.timings = t
    return r


def _shift(findings, deltas):
    """After the remediation pass, move first-pass placeholder offsets by the length change of every
    remediation edit that lies before them."""
    for f in findings:
        d = sum(dl for s, dl in deltas if s < f.red_start)
        f.red_start += d
        f.red_end += d
    return findings


def _summ(v: dict) -> dict:
    """Verification summary with no raw values (types and counts only)."""
    def types(lst):
        out = {}
        for f in lst:
            out[f.entity_type] = out.get(f.entity_type, 0) + 1
        return out
    return {"residual_count": v["residual_count"], "rescan": types(v["rescan"]), "leaks": len(v["leaks"]),
            "cross_view": types(v["cross_view"])}


def run_batch(paths, settings: Settings = DEFAULT, progress=print) -> list:
    results = []
    for p in paths:
        progress(f"processing {Path(p).name} ...")
        r = run_file(p, settings)
        progress(f"  -> {r.status.value} ({sum(r.timings.values()):.0f}s, {len(r.findings)} findings)")
        results.append(r)
    return results


if __name__ == "__main__":
    from .report import export, summary_table
    res = run_batch(sys.argv[1:])
    print(summary_table(res).to_string())
    print("exported to", export(res, "outputs"))

In [ ]:
%%writefile cadence_pii/redact.py
"""Offset-based redaction with stable, typed placeholders.

The same value (normalised) always gets the same placeholder within one document, e.g. every
mention of one e-mail address becomes [EMAIL_2]. This is pseudonymisation for readability, not
anonymisation: the mapping exists only in memory and is never exported.
"""
from __future__ import annotations

import re

PLACEHOLDER_RX = re.compile(r"\[[A-Z_]+_\d+\]")


def _norm(etype: str, v: str) -> str:
    if etype in ("PHONE", "US_SSN", "FINANCIAL_ACCOUNT"):
        return re.sub(r"\D", "", v)
    return re.sub(r"[^a-z0-9]", "", v.lower())


def redact(text: str, findings: list, mapping: dict = None) -> tuple:
    """Return (redacted_text, mapping). Findings must be non-overlapping (merge.resolve guarantees it)."""
    mapping = {} if mapping is None else mapping
    counters: dict = {}
    for (etype, _k), ph in mapping.items():
        n = int(ph.rsplit("_", 1)[1].rstrip("]"))
        counters[etype] = max(counters.get(etype, 0), n)
    out, pos, cur = [], 0, 0
    for f in sorted(findings, key=lambda f: f.start):
        if f.start < pos:  # defensive: overlapping input
            continue
        key = (f.entity_type, _norm(f.entity_type, f.value))
        if key not in mapping:
            counters[f.entity_type] = counters.get(f.entity_type, 0) + 1
            mapping[key] = f"[{f.entity_type}_{counters[f.entity_type]}]"
        f.placeholder = mapping[key]
        out.append(text[pos:f.start])
        cur += f.start - pos
        f.red_start, f.red_end = cur, cur + len(f.placeholder)
        out.append(f.placeholder)
        cur += len(f.placeholder)
        pos = f.end
    out.append(text[pos:])
    return "".join(out), mapping


def masked_context(redacted: str, f, width: int = 60) -> str:
    """Shape-only snippet: verification can miss PII in neighbouring free text.

Preserve placeholder tags and spacing, but mask every other non-space character.
This keeps the audit safe even for REVIEW_REQUIRED text with OCR-corrupted PII.
"""
    if f.red_start < 0:
        return ""
    a, b = max(0, f.red_start - width), min(len(redacted), f.red_end + width)
    snippet = redacted[a:b]
    shaped = re.sub(r"\[[A-Z_]+_\d+\]|[^\s]", lambda m: m.group() if PLACEHOLDER_RX.fullmatch(m.group()) else "•", snippet)
    return ("…" if a else "") + shaped.replace("\n", " ⏎ ") + ("…" if b < len(redacted) else "")

In [ ]:
%%writefile cadence_pii/report.py
"""Audit reporting and exports. Reports contain types, counts, locators and masked context only."""
from __future__ import annotations

import html
import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from .config import ENTITY_TYPES
from .models import Status
from .redact import masked_context


def summary_table(results) -> pd.DataFrame:
    rows = []
    for r in results:
        c = Counter(f.entity_type for f in r.findings)
        d = r.document
        row = {"file": r.filename, "type": r.filetype, "status": r.status.value,
               "pages/slides": (d.inventory.get("pages_or_slides") if d else None),
               "blocks": len(d.blocks) if d else 0, "PII findings": len(r.findings)}
        row.update({t: c.get(t, 0) for t in ENTITY_TYPES})
        row.update({"review reasons": len(r.reasons), "residual after verify":
                    r.verification.get("final", {}).get("residual_count"), "secs": round(sum(r.timings.values()), 1)})
        rows.append(row)
    return pd.DataFrame(rows)


def findings_table(r, limit=None) -> pd.DataFrame:
    rows = []
    for f in sorted(r.findings, key=lambda f: (f.red_start if f.red_start >= 0 else 10**12)):
        p = f.public(masked_context(r.redacted_text, f))
        rows.append({k: p[k] for k in ("entity_type", "placeholder", "locator", "method", "recognizer", "score",
                                       "value_masked", "context_masked")})
    df = pd.DataFrame(rows)
    return df.head(limit) if limit else df


def audit_record(r) -> dict:
    d = r.document
    return {
        "file": r.filename, "file_type": r.filetype, "sha256": r.sha256, "size_bytes": r.size_bytes,
        "status": r.status.value, "reasons": r.reasons, "errors": [e for e in r.errors if "Traceback" not in e],
        "inventory": d.inventory if d else {}, "extraction_warnings": d.warnings if d else [],
        "counts_by_type": dict(Counter(f.entity_type for f in r.findings)),
        "counts_by_method": dict(Counter(f.method for f in r.findings)),
        "verification": r.verification, "timings_s": r.timings, "review_log": r.review_log,
        "findings": [f.public(masked_context(r.redacted_text, f)) for f in r.findings],
    }


def export(results, outdir="outputs") -> Path:
    """Write redacted text for every processed file, AI-ready copies only for CLEARED files, plus audit files."""
    out = Path(outdir)
    (out / "redacted").mkdir(parents=True, exist_ok=True)
    (out / "ai_ready").mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for r in results:
        stem = Path(r.filename).stem
        if r.redacted_text:
            header = f"<!-- {r.filename} | status {r.status.value} | redacted {stamp} -->\n\n"
            (out / "redacted" / f"{stem}.redacted.md").write_text(header + r.redacted_text, encoding="utf-8")
        ai = out / "ai_ready" / f"{stem}.md"
        if r.status == Status.CLEARED:
            ai.write_text(r.redacted_text, encoding="utf-8")
        elif ai.exists():
            ai.unlink()  # never leave a stale release behind
    audit = {"generated_at": stamp, "files": [audit_record(r) for r in results]}
    (out / "audit_report.json").write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
    rows = []
    for r in results:
        for f in r.findings:
            p = f.public(masked_context(r.redacted_text, f))
            rows.append({"file": r.filename, **{k: p[k] for k in ("entity_type", "placeholder", "locator", "page",
                                                                  "start", "end", "method", "recognizer", "score",
                                                                  "value_masked", "context_masked")}})
    pd.DataFrame(rows).to_csv(out / "findings.csv", index=False)
    (out / "summary.html").write_text(_html(results, stamp), encoding="utf-8")
    return out


def _html(results, stamp) -> str:
    colors = {"CLEARED": "#1a7f37", "REVIEW_REQUIRED": "#9a6700", "BLOCKED": "#cf222e", "ERROR": "#6e7781"}
    parts = [f"<h1>Cadence PII scan - audit summary</h1><p>Generated {stamp}. No raw PII values are included.</p>",
             summary_table(results).to_html(index=False)]
    for r in results:
        parts.append(f"<h2>{html.escape(r.filename)} - <span style='color:{colors.get(r.status.value)}'>"
                     f"{r.status.value}</span></h2>")
        if r.reasons:
            parts.append("<ul>" + "".join(f"<li>{html.escape(x)}</li>" for x in r.reasons) + "</ul>")
        if r.findings:
            parts.append(findings_table(r).to_html(index=False))
    return "<html><head><meta charset='utf-8'><style>body{font-family:sans-serif;margin:2em}table{border-collapse:collapse;font-size:12px}td,th{border:1px solid #ccc;padding:3px 6px}</style></head><body>" + "".join(parts) + "</body></html>"

In [ ]:
%%writefile cadence_pii/verify.py
"""Independent verification of redacted output.

Three checks, each designed to catch a different failure mode:
1. Re-scan: regex + Presidio (lower threshold) on the redacted text, ignoring placeholders.
2. Leak check: every value that was redacted must not still appear verbatim elsewhere.
3. Cross-view (PDF): PII detected in the *independent* full-resolution Tesseract reading must not
   appear in the redacted text (catches values that Docling's OCR read differently or that
   detection missed in the primary view).
Passing these checks reduces risk; it does not prove zero residual PII.
"""
from __future__ import annotations

import re

from .config import DEFAULT, Settings
from .detect.run import detect_text, finalize
from .models import Finding
from .redact import PLACEHOLDER_RX

STRUCTURED = {"EMAIL", "PHONE", "EMPLOYEE_ID", "US_SSN", "PASSPORT", "TAX_ID", "FINANCIAL_ACCOUNT",
              "DATE_OF_BIRTH", "ADDRESS"}


def _outside_placeholders(text: str, f: Finding) -> bool:
    for m in PLACEHOLDER_RX.finditer(text, max(0, f.start - 40), f.end + 40):
        if m.start() <= f.start and f.end <= m.end():
            return False
    return not PLACEHOLDER_RX.fullmatch(f.value.strip())


def rescan(redacted: str, settings: Settings = DEFAULT) -> list:
    raw = detect_text(redacted, settings, presidio_threshold=settings.verify_presidio_threshold,
                      segments=_segments(redacted))
    fs = finalize(raw, redacted)
    out = []
    for f in fs:
        # finalize() unions overlapping spans; trim placeholders back out of the residual
        if not _outside_placeholders(redacted, f):
            continue
        if PLACEHOLDER_RX.search(f.value):
            continue
        out.append(f)
    return out


def _segments(text: str, size: int = 20000) -> list:
    """Split long text on paragraph boundaries for spaCy."""
    segs, s = [], 0
    while s < len(text):
        e = min(len(text), s + size)
        if e < len(text):
            k = text.rfind("\n\n", s, e)
            e = k if k > s else e
        segs.append((s, e))
        s = e
    return segs


def _leak_eligible(f) -> bool:
    """Only high-precision values are swept document-wide; a single NER token could be a common word."""
    v = f.value.strip()
    if len(v) < 4:
        return False
    if f.entity_type != "PERSON":
        return True
    return len(v.split()) >= 2 or "regex" in f.method


def leak_check(redacted: str, findings: list) -> list:
    out = []
    for v in sorted({f.value.strip() for f in findings if _leak_eligible(f)}):
        for m in re.finditer(re.escape(v), redacted):
            out.append(Finding("LEAK", m.start(), m.end(), v, 1.0, "leak_check", "verbatim_value"))
    return out


def cross_view(redacted: str, alt_text: str, settings: Settings = DEFAULT) -> list:
    if not alt_text:
        return []
    alt = finalize(detect_text(alt_text, settings, segments=_segments(alt_text)), alt_text)
    out, seen = [], set()
    for f in alt:
        v = f.value.strip()
        if len(v) < 4 or v in seen:
            continue
        seen.add(v)
        for m in re.finditer(re.escape(v), redacted):
            g = Finding(f.entity_type, m.start(), m.end(), v, f.score, "cross_view", f.recognizer)
            if _outside_placeholders(redacted, g):
                out.append(g)
    return out


def verify(redacted: str, findings: list, alt_text: str = "", settings: Settings = DEFAULT) -> dict:
    res = rescan(redacted, settings)
    leaks = leak_check(redacted, findings)
    xv = cross_view(redacted, alt_text, settings) if alt_text else []
    return {"rescan": res, "leaks": leaks, "cross_view": xv,
            "residual_count": len(res) + len(leaks) + len(xv)}

In [ ]:
# Stop early if Docling cannot be imported in this runtime.
# The pipeline has a safe fallback, but this case study's PDF/DOCX structure results use Docling.
from cadence_pii.extract.docling_ext import _converter
_converter('eng')
print('Docling extractor ready')

## Upload and inventory

Colab prompts for PDF, DOCX and PPTX. Locally, set `CADENCE_INPUT_DIR` to a folder of supported files, or place the supplied files in `Case Study 2 Supporting Material`. The local path is useful for a privacy-restricted walkthrough.

In [ ]:
from IPython.display import display
from cadence_pii.ingest import validate
from cadence_pii.config import DEFAULT
SUPPORTED = {'.pdf', '.docx', '.pptx'}
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    input_dir = Path('cadence_inputs')
    input_dir.mkdir(exist_ok=True)
    for name, contents in uploaded.items():
        safe_name = Path(name).name
        (input_dir / safe_name).write_bytes(contents)
        Path(safe_name).unlink(missing_ok=True)  # remove duplicate left by Colab upload
else:
    input_dir = Path(os.environ.get('CADENCE_INPUT_DIR', 'Case Study 2 Supporting Material'))
paths = sorted(p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)
if not paths:
    raise FileNotFoundError(f'No PDF/DOCX/PPTX files found in {input_dir}')
for path in paths:
    try:
        meta = validate(path, DEFAULT)
        print(path.name, meta['filetype'], meta['size_bytes'], 'bytes')
    except Exception as exc:
        print(path.name, 'validation issue:', type(exc).__name__, str(exc))

## Extract, detect, redact, verify and gate

The PDF uses layout-aware extraction plus OCR of the full pages and figures. The second OCR reading checks for missed exact PII values. PPTX uses its native shapes; DOCX uses layout extraction plus image OCR. Regex and Presidio findings are merged and replaced with typed placeholders. Any detector failure, unresolved residual or uncertain image extraction stops release.

In [ ]:
from cadence_pii.pipeline import run_batch
from cadence_pii.report import summary_table, findings_table
results = run_batch(paths)
display(summary_table(results))
for result in results:
    print('\n', result.filename, result.status.value)
    if result.document:
        print('inventory:', result.document.inventory)
    print('review reasons:', *result.reasons[:8], sep='\n  ')
    display(findings_table(result, limit=12))

## Safe preview and release boundary

`SHOW_RAW` defaults to false. The left column describes the original structure without printing extracted source text; the right column shows a redacted excerpt. The masked findings table above supplies source locators and evidence shapes. Turn on raw preview only in a private session if a reviewer explicitly needs it.

In [ ]:
from cadence_pii.gate import ReleaseBlocked, get_clean_text, approve
from cadence_pii.redact import masked_context
SHOW_RAW = False
for result in results:
    print('\nFILE:', result.filename)
    if result.document:
        print('ORIGINAL:', len(result.document.blocks), 'blocks;', len(result.document.text), 'characters; raw hidden')
        if SHOW_RAW:
            print('RAW PREVIEW:', result.document.text[:600])
    preview = masked_context(result.redacted_text, result.findings[0], width=180) if result.findings else '[no findings; raw hidden]'
    print('MASKED REDACTED PREVIEW:', preview)
    print('VERIFY:', result.verification.get('final', {}), 'STATUS:', result.status.value)
    try:
        clean = get_clean_text(result)
        print('Release boundary: passed;', len(clean), 'redacted characters available')
    except ReleaseBlocked as exc:
        print('Release boundary:', exc)

## Human sign-off, only after source review

To release a review-required file, inspect every review reason and the original image regions first. Enter a reviewer name and a specific note below, then rerun this cell. Blocked and error files cannot be approved. Signing off does not rerun detection; it records accountable human review of the stated uncertainty.

In [ ]:
REVIEWER = ''
REVIEW_NOTE = ''
if REVIEWER.strip() and REVIEW_NOTE.strip():
    for result in results:
        if result.status.value == 'REVIEW_REQUIRED':
            approve(result, REVIEWER, REVIEW_NOTE)
            print(result.filename, 'approved; released chars:', len(get_clean_text(result)))
else:
    print('No sign-off entered; review-required files remain withheld.')

## Export and cleanup

The ZIP contains redacted text, a masked audit report and `ai_ready` copies for cleared files only. Source uploads are excluded. Keep the archive private: redacted text can still contain OCR misses. The last cell can remove uploaded input copies after you have finished reviewing the source.

In [ ]:
from cadence_pii.report import export
out_dir = export(results, 'cadence_outputs')
archive = shutil.make_archive('Cadence_PII_Results', 'zip', root_dir=out_dir)
print('Export:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)

In [ ]:
CLEANUP_UPLOADED_INPUTS = False
if CLEANUP_UPLOADED_INPUTS and IN_COLAB:
    for path in paths:
        path.unlink(missing_ok=True)
    print('Deleted uploaded input copies from this runtime. Restart the runtime to clear in-memory content.')
else:
    print('Input copies retained for review; set CLEANUP_UPLOADED_INPUTS=True to delete Colab copies.')

## Limits

The annotated sample is small and in-sample. The PDF form photo on page 18 is illegible to OCR. OCR corruption inside an identifier can evade both passes; the second reading catches exact matches only. Vector images may be unreadable. Common-word names can be missed. Review the audit reasons and source images before authorising a downstream model. The notebook has been verified locally; a Colab run is pending.